# BR-CP — Corrected CPU Audit after GPU experiment (11 November 2026)

**Do not rerun GPU inference.** This notebook uses the already saved Chronos-Bolt forecast caches to correct the off-by-one conformal quantile, select an expanded data-space detector threshold on development-only series, replay all nine calibration methods, and recompute descriptive coverage debt over the first 100 post-intervention observations (before the engineered pulse ends at 120).

This is a **post-hoc scientific correction**, not an independently pre-registered confirmation. It does not establish model-training disjointness or guarantee distribution-free coverage in time series.

Select **CPU** as the runtime; GPU is unnecessary. Run the cells in order. Existing original GPU results are left unchanged.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
SOURCE_ZIP=Path('/content/drive/MyDrive/BRCP_Recovery_2026-11-10/BRCP_Chronos_GPU_CODE_FIX_AND_AUDIT_2026-11-11.zip')
CACHE_ROOT=Path('/content/drive/MyDrive/BRCP_Chronos_Bolt_GPU_Confirm_2026-11-11/e6247c5f3e92')
OUTPUT=Path('/content/drive/MyDrive/BRCP_GPU_AUDIT_FIXED_2026-11-11')
assert SOURCE_ZIP.exists(), 'Corrected audit ZIP not found in the recovery folder'
assert (CACHE_ROOT/'RUN_MANIFEST.json').exists(), 'Original GPU run manifest not found'
assert len(list((CACHE_ROOT/'forecast_cache').glob('*.npz')))>=108, 'Cached original predictions are incomplete'
OUTPUT.mkdir(exist_ok=True,parents=True)
print('Original GPU cache:',CACHE_ROOT)
print('Fixed audit destination:',OUTPUT)

In [ ]:
import zipfile, sys, subprocess
code_root=Path('/content/brcp_scientific_fix')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    source_files=[name for name in z.namelist() if name.startswith('source/')]
    for name in source_files:
        z.extract(name,code_root)
script=code_root/'source/audit_cached.py'
assert script.exists(),script
print('Using audited source:',script)
subprocess.check_call([sys.executable,'-m','pip','install','-q','numpy','pandas','scipy'])

In [ ]:
# CPU-only: replays cached forecasts; no model download, no Chronos GPU execution.
import subprocess
proc=subprocess.run([sys.executable,str(script),
    '--input-root',str(CACHE_ROOT),'--output-root',str(OUTPUT),'--workers','2'],check=True)
assert proc.returncode==0
print('CORRECTED CPU AUDIT COMPLETED')

In [ ]:
# Verify outputs before any scientific claims.
import pandas as pd, json
status=json.loads((OUTPUT/'SCIENTIFIC_STATUS.json').read_text())
assert status['n_method_rows']==972
assert status['n_series_scenarios']==108
print('Thresholds selected on development:',status['development_thresholds'])
print('Test empirical alarm rates:',status['test_empirical_alarm_rates'])
summary=pd.read_csv(OUTPUT/'summary_CORRECTED.csv')
print(summary[summary.method.isin(['ACI(0.05)','BRCP','BRCP-data'])].to_string(index=False))
cd=pd.read_csv(OUTPUT/'cd100_descriptive_CORRECTED.csv')
print('Exploratory CD100 (not a formal TTR result):')
print(cd[cd.method.isin(['ACI(0.05)','BRCP','BRCP-data'])].to_string(index=False))

In [ ]:
# Export shareable results, preserving original GPU caches untouched.
import hashlib, zipfile
archive=OUTPUT.parent/'BRCP_Corrected_CPU_Audit_Evidence_2026-11-11.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for filename in ['SCIENTIFIC_STATUS.json','development_threshold_grid.csv',
       'all_method_results_CORRECTED.csv','summary_CORRECTED.csv','within_series_ratios_CORRECTED.csv',
       'cd100_descriptive_CORRECTED.csv','cd100_curves_CORRECTED.csv']:
        p=OUTPUT/filename
        assert p.exists(),p
        z.write(p,filename)
    z.write(SOURCE_ZIP,arcname=SOURCE_ZIP.name)
print('Results ZIP in Drive:',archive)
print('SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
print('Keep the old GPU evidence as an immutable original; use these corrected results for subsequent manuscript revisions.')